# 03 - Keyword Mapping
Consolidation and generation of the master tourism keyword list for search volume extraction via Google Ads Planner (DataForSEO).

## 1. Imports

In [88]:
from pathlib import Path
from unidecode import unidecode
import pandas as pd

## 2. Paths Configuration

In [89]:
INTERIM_PATH = Path("../data/interim")
REVIEWED_PATH = Path("../data/interim/reviewed")
PROCESSED_PATH = Path("../data/processed")

REVIEWED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

INTERIM_PATH.mkdir(
    parents=True,
    exist_ok=True
)

## 3. Data Ingestion

In [90]:
df_azulmarino = pd.read_excel(
    REVIEWED_PATH / "azulmarino_destinations_reviewed.xlsx"
)

df_tempsdoci = pd.read_excel(
    REVIEWED_PATH / "tempsdoci_destinations_reviewed.xlsx"
)

df_countries_master = pd.read_parquet(
    INTERIM_PATH / "01_countries_ine_master_v2.parquet"
)

## 4. Normalization Methods

In [91]:
def normalize_text(text):
    """
    Standardizes text by converting to lowercase, removing accents and stripping whitespace.
    Args: text (str): The raw text string to normalize.
    Returns: str or None: The cleaned string, or None if the input is missing (NaN).
    """
    if pd.isna(text):
        return None

    text = str(text).lower().strip()
    text = unidecode(text)
    text = text.replace("-", " ")

    return text

In [92]:
def normalize_columns(df):
    """
    Standardizes DataFrame column names by converting to lowercase, 
    stripping whitespace and replacing spaces with underscores.
    Args: df (pd.DataFrame): The DataFrame to process.
    Returns: pd.DataFrame: The DataFrame with normalized column names.
    """
    df = df.copy()

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )

    return df

## 5. Apply Data Normalization

Standardizing column names across all imported datasets before integration to ensure consistency.

In [93]:
df_azulmarino = normalize_columns(df_azulmarino)
df_tempsdoci = normalize_columns(df_tempsdoci)
df_countries_master = normalize_columns(df_countries_master)

## 6. Official Keywords Preparation

In [94]:
df_keywords_ine = pd.DataFrame({

    "raw_title":
        df_countries_master["country"],

    "search_term":
        df_countries_master["country"]
        .map(normalize_text),

    "parent_country":
        df_countries_master["country"]
        .map(normalize_text),

    "keyword_type":
        "country",

    "source":
        "ine",

    "url":
        None
})

## 7. Filter Reviewed Records & Column Selection

Filter out records marked for deletion during manual review

In [95]:
if "delete" in df_azulmarino.columns:

    df_azulmarino = (
        df_azulmarino[
            df_azulmarino["delete"] != True
        ]
    )

if "delete" in df_tempsdoci.columns:

    df_tempsdoci = (
        df_tempsdoci[
            df_tempsdoci["delete"] != True
        ]
    )

### Columns' selection

In [96]:
KEEP_COLUMNS = [

    "raw_title",
    "search_term",
    "parent_country",
    "keyword_type",
    "source",
    "url"
]

In [97]:
df_azulmarino = (df_azulmarino[KEEP_COLUMNS])
df_tempsdoci = (df_tempsdoci[KEEP_COLUMNS])

## 8. Dataset Integration

In [98]:
df_keyword_mapping = pd.concat([

    df_keywords_ine,
    df_azulmarino,
    df_tempsdoci

], ignore_index=True)

Normalize core columns post-integration

In [99]:
df_keyword_mapping["search_term"] = (
    df_keyword_mapping["search_term"]
    .map(normalize_text)
)

df_keyword_mapping["parent_country"] = (
    df_keyword_mapping["parent_country"]
    .map(normalize_text)
)

## 9. Multi-Country Treatment

Some commercial terms represent regions or routes spanning multiple countries.
In these cases, the search term is replicated and assigned to each involved country 
to preserve the regional demand signal and facilitate aggregations by country.

Examples:
* Bálticos (Baltics) → Estonia, Letonia, Lituania
* Indochina → Vietnam, Camboya, Laos
* Cáucaso (Caucasus) → Armenia, Georgia, Azerbaiyán
* Balcanes (Balkans) → Montenegro, Albania, Croacia


### ⚠ Methodological Note: Multi-Country Term Aggregation

Regional terms (e.g. "balcanes", "bálticos", "indochina") are replicated across all associated countries to preserve the regional demand signal.

**Known limitation:** This means the full search volume for a regional term is attributed to each constituent country independently — it is **not divided** among them. This may artificially inflate search signal for smaller countries within a region (e.g. Montenegro, Albania within the Balkans).

**Justification:** Splitting volume equally across countries would imply a uniform distribution of traveler interest that is equally unverifiable. Replication with explicit documentation is preferred in this case.

**Consequence for downstream analysis:** When aggregating search volumes 
by country, regional multi-country terms should be interpreted as *regional demand indicators*, not as country-level demand estimates.
Correlations for small countries with high multi-country term exposure (Balkans, Baltics, Caucasus) should be interpreted with extra caution.

### Expand terms assigned to multiple countries separated by semicolons

In [100]:
df_keyword_mapping["parent_country"] = (
    df_keyword_mapping["parent_country"]
    .astype(str)
    .str.split(";")
)

In [101]:
df_keyword_mapping = (
    df_keyword_mapping
    .explode("parent_country")
)

In [102]:
df_keyword_mapping["parent_country"] = (

    df_keyword_mapping["parent_country"]
    .str.strip()
)

## 10. Priority-Based Deduplication

In case of overlapping keywords between sources, priority is given to official entities (INE), 
followed by competitors based on market relevance.

In [103]:
records_before_dedup = len(df_keyword_mapping)

In [104]:
source_priority = {

    "ine": 1,
    "azulmarino": 2,
    "tempsdoci": 3
}

In [105]:
df_keyword_mapping["priority"] = (
    df_keyword_mapping["source"]
    .map(source_priority)
)

In [106]:
# Sort by priority and drop duplicates keeping the highest priority source
df_keyword_mapping = (
    df_keyword_mapping
    .sort_values("priority")
    .drop_duplicates(
        subset=["search_term", "parent_country"],
        keep="first"
    )
    .drop(columns="priority")
)

print(f"Records before dedup: {records_before_dedup}")
print(f"Records after dedup: {len(df_keyword_mapping)}")
print(f"Dropped: {records_before_dedup - len(df_keyword_mapping)}")

Records before dedup: 395
Records after dedup: 236
Dropped: 159


In [107]:
# Final cleanup
df_keyword_mapping = (
    df_keyword_mapping
    .drop_duplicates()
    .sort_values(["parent_country", "search_term"])
    .reset_index(drop=True)
)



## 11. Country Validation

Validate that all parent_country values in the keyword mapping have a corresponding country in the INE master dataset.
Orphans = keywords that cannot be joined to mobility data.

In [108]:
ine_countries = set(df_keywords_ine["parent_country"])
mapping_countries = set(df_keyword_mapping["parent_country"].dropna())
orphans = mapping_countries - ine_countries

print(f"INE countries: {len(ine_countries)}")
print(f"Mapping countries: {len(mapping_countries)}")
print(f"Orphans (no INE match): {len(orphans)}")

if orphans:
    print("\n--- ORPHAN COUNTRIES ---")
    print(orphans)
else:
    print("\n✔ All parent_country values match the INE master dataset.")

INE countries: 155
Mapping countries: 166
Orphans (no INE match): 11

--- ORPHAN COUNTRIES ---
{'belice', 'polinesia', 'groenlandia', 'reunion', 'guinea bissau', 'sudan', 'trinidad & tobago', 'taiwan', 'zambia', 'balcanes', 'puerto rico'}


In [109]:
# Check if orphans exist under alternative spellings in INE
orphan_list = sorted(list(orphans))
print(list(orphan_list))

['balcanes', 'belice', 'groenlandia', 'guinea bissau', 'polinesia', 'puerto rico', 'reunion', 'sudan', 'taiwan', 'trinidad & tobago', 'zambia']


In [110]:
# Expected orphans after spelling corrections:
KNOWN_ORPHANS = {
    'balcanes',        # regional multi-country term, expected
    'puerto rico',     # non-sovereign territory (USA)
    'reunion',         # non-sovereign territory (France)
    'groenlandia',     # non-sovereign territory (Denmark)
    'polinesia',       # non-sovereign territory (France)
    'belice',          # not covered by INE dataset
    'sudan',           # not covered by INE dataset
    'taiwan',          # not covered by INE dataset
    'trinidad & tobago', # not covered by INE dataset
    'zambia',          # not covered by INE dataset
    'guinea bissau',   # not covered by INE dataset
}

unexpected_orphans = orphans - KNOWN_ORPHANS
if unexpected_orphans:
    print(f"⚠ Unexpected orphans found: {unexpected_orphans}")
else:
    print("✔ All orphans are accounted for and documented.")

✔ All orphans are accounted for and documented.


## 12. Final Data Review

In [111]:
print(f"Dataset Shape: {df_keyword_mapping.shape}\n")
print("--- NULL VALUES ---")
print(df_keyword_mapping.isnull().sum())

df_keyword_mapping.head()

Dataset Shape: (236, 6)

--- NULL VALUES ---
raw_title           0
search_term         0
parent_country      0
keyword_type        0
source              0
url               156
dtype: int64


,raw_title,search_term,parent_country,keyword_type,source,url
0,Albania,albania,albania,country,ine,None
1,"ALBANIA, MACEDONIA DEL NORTE Y CORFÚ: Balcanes...",balcanes,albania,region,tempsdoci,/viajes/viaje-en-grupo-albania-macedonia-corfu...
2,Alemania,alemania,alemania,country,ine,None
3,Andorra,andorra,andorra,country,ine,None
4,Angola,angola,angola,country,ine,None


## 13. Master Dataset Export

In [112]:
df_keyword_mapping.to_parquet(
    PROCESSED_PATH / "03_keyword_mapping_master_v2.parquet"
)

df_keyword_mapping.to_excel(
    PROCESSED_PATH / "03_keyword_mapping_master_v2.xlsx",
    index=False
)

## 14. Next Steps

Use the `03_keyword_mapping_master_v2` dataset for:
* Search volume extraction via DataForSEO Google Ads API
* Google Trends signal extraction and normalization
* Semantic aggregation of travel demand by country
* Correlation analysis against INE mobility data